In [1]:
with open(r'C:\Users\User\my\py\NLP\mayakovsky_poems.txt', 'r', encoding='utf-8') as file:
    text = file.read()

In [2]:
import re

# text = re.sub(r'[A-Za-z]', '', text)
def normalize_text(text):
    replacements = {
    # Кавычки и тире
    '“': '"', '”': '"',
    '‘': "'", '’': "'",
    '—': '-', '–': '-',
    '…': '...',
    '«': '"', '»': '"',

    # Невидимые пробелы и неразрывные
    '\u2003': ' ',
    '\u2004': ' ',
    '\u200e': ' ',
    '\xa0': ' ',
    '\u202f': ' ',
    '\u00ad': '',  # soft hyphen

    # Экзотика
    '×': '',
    '½': '',
    '̀': '',
    '́': '',
    '·': '',
    '№': '',
    'à': '', 'é': '', 'ö': '', 'ü': '',
    'ѣ': '',
    'Ё': '',

    # Технические символы
    '=': '', '/': '', '[': '', ']': '', '{': '', '}': '',
    '<': '', '>': '', '_': '', '%': '', ';': '', '*': '',

    # Английские буквы (удаление)
    **{ch: '' for ch in 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'}
}
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text
text=normalize_text(text)

Нормализация была изменена на rmsnorm, что должно давать больше стабьильности и требовать меньше параметров. Также был добавлен ROPE, позволяет задавать позиции использовать значение векторов в пространстве. Добавлена MoE, позволяет создать несколько экспетров, в моем случае- 4, после чего происходит отбор экспертов для обработки для каждого токена.   

In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F

from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

# hyperparameters
batch_size = 16         
block_size = 128        
n_embd = 256           
n_head = 4              
n_layer = 6             
dropout = 0.1           
learning_rate = 3e-4   
max_iters = 10000       
eval_interval = 100     
eval_iters = 100        
patience = 10
torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'



# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        desc = f"[{split.upper()} EVAL]"
        for k in tqdm(range(eval_iters), desc=desc, leave=False):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# Сделаем нормализация RMSnorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.norm(2, dim=-1, keepdim=True)
        rms = norm / (x.size(-1) ** 0.5)
        return self.scale * x / (rms + self.eps)

# Реализация RoPE 
def apply_rope(x):
    # x: (B, T, H, D)
    B, T, H, D = x.shape
    assert D % 2 == 0
    half = D // 2

    freqs = 1.0 / (10000 ** (torch.arange(0, half, device=x.device).float() / half))
    pos = torch.arange(T, device=x.device).float()
    angles = torch.einsum('i,j->ij', pos, freqs)
    sin, cos = angles.sin(), angles.cos()
    sin = sin[None, :, None, :]
    cos = cos[None, :, None, :]

    x1, x2 = x[..., :half], x[..., half:]
    x_rot = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return x_rot


class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size

    def forward(self, x):
        B, T, _ = x.shape
        H = 1
        D = self.head_size

        # produce q, k, v of shape (B, T, 1, D)
        q = self.query(x).view(B, T, H, D)
        k = self.key(x).view(B, T, H, D)
        v = self.value(x).view(B, T, H, D)

        q = apply_rope(q)
        k = apply_rope(k)

        att = torch.einsum('bthd,bshd->bhts', q, k) * (D ** -0.5)
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        out = torch.einsum('bhts,bshd->bthd', att, v).reshape(B, T, D)
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))
# Реализуем класс MoEFeedForward
class MoEFeedForward(nn.Module):
    #буду использовать 4 эксперта
    def __init__(self, n_embd, n_experts=4):
        super().__init__()
        self.n_experts = n_experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(n_embd, 4 * n_embd),
                nn.ReLU(),
                nn.Linear(4 * n_embd, n_embd)
            )
            for _ in range(n_experts)
        ])
        self.gate = nn.Linear(n_embd, n_experts)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, T, C)
        B, T, C = x.shape
        gate_scores = F.softmax(self.gate(x), dim=-1)  # (B, T, E)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=-1)  # (B, T, C, E)
        # weighted sum: einsum over expert dim
        output = torch.einsum('bte,btce->btc', gate_scores, expert_outputs)
        return self.dropout(output)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = MoEFeedForward(n_embd, n_experts=4)  # Moe
        self.ln1 = RMSNorm(n_embd)
        self.ln2 = RMSNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))  # MoE вместо FF
        return x


class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx)  # (B, T, C)
        x = self.blocks(x)  # (B, T, C)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            return logits, None
        else:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
            return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = CosineAnnealingLR(optimizer, T_max=max_iters)

best_val_loss = float('inf')
patience_counter = 0


##Добавим early stopping
for iter in range(max_iters):
    if iter % eval_interval == 0:
        print(f"[INFO] Starting evaluation at step {iter}")
        losses = estimate_loss()
        val_loss = losses['val']
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
            print("✅ New best model saved.")
        else:
            patience_counter += 1
            print(f"⚠️ No improvement. Patience left: {patience - patience_counter}")
            if patience_counter >= patience:
                print(f"⏹️ Early stopping at step {iter}")
                break

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

model.load_state_dict(torch.load('best_model.pt'))
model.eval()
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

14.242159 M parameters
[INFO] Starting evaluation at step 0


step 0: train loss 4.5400, val loss 4.4489
✅ New best model saved.
[INFO] Starting evaluation at step 100


step 100: train loss 2.5858, val loss 2.5480
✅ New best model saved.
[INFO] Starting evaluation at step 200


step 200: train loss 2.4104, val loss 2.2694
✅ New best model saved.
[INFO] Starting evaluation at step 300


step 300: train loss 2.3094, val loss 2.2181
✅ New best model saved.
[INFO] Starting evaluation at step 400


...

step 9400: train loss 1.4619, val loss 1.6191
⚠️ No improvement. Patience left: 3
[INFO] Starting evaluation at step 9500


step 9500: train loss 1.4556, val loss 1.6086
⚠️ No improvement. Patience left: 2
[INFO] Starting evaluation at step 9600


step 9600: train loss 1.4641, val loss 1.6220
⚠️ No improvement. Patience left: 1
[INFO] Starting evaluation at step 9700


C:\Users\User\AppData\Local\Temp\ipykernel_8728\2815699856.py:246: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pt'))


step 9700: train loss 1.4573, val loss 1.6370
⚠️ No improvement. Patience left: 0
⏹️ Early stopping at step 9700
	ни глаз, 	блокад закрикать ее.  	набольше! Внимание решь  		и хастро-таки: - Мы ткорою  	я господа-богу! Замечет: "Вот сегодня  	резен кручилы! Когда бежал себе 	ужасый?"  ) Молчали  	ученые нее. Слышите к журнальные зря. Сработаешь  		наплаченными смеется, в семансом -  			господина...   	Милюки! Тридце тИСКАРИПА ТЕРАРОННАУ Авца - шелка!" - 		ни года. Бечет воскрёса - точка, и я в ногу другу - 	и буржуазий человечеством съел. Так о голодающему требуешь - и та карман две такого, тоже наш потребовать сколько чаще не жена аляпинская уха... Коминтерные точно руки, а говорят: "Мы сейчас же, Но, между я с Антанты мне нам, чтоб перс. Призывом мигом "с прикрычкам с нами для бабы пашни толпы везде ходящие" (тела меня бюрократа), но поп (им не поискашнего его пятимного, чтобы мячить ребята?! Понемножко былу тебя построенный! Не так у нас могут сфать ухоть вороньями с водой, бросится

Для улучшения качества loss, необходимо число эмбедингов 512 и более, количество слоев более 12. Однако обработка такой модели в моем случае займет в десятки раз больше времени. Более того, применение Top-1 routing с load balancing могло бы улучшить качество предсказаний. Это бы позволило добавить регуляризацию и экономить ресурсы, поскольку для каждого токена выбирался бы только 1 эксперт.

In [4]:
#SwitchMoEFeedForward
import torch
import torch.nn as nn
from torch.nn import functional as F

from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

# hyperparameters
batch_size = 16         
block_size = 128        
n_embd = 256           
n_head = 4              
n_layer = 6             
dropout = 0.1           
learning_rate = 3e-4   
max_iters = 10000       
eval_interval = 100     
eval_iters = 100        
patience = 10
torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'



# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        desc = f"[{split.upper()} EVAL]"
        for k in tqdm(range(eval_iters), desc=desc, leave=False):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# Сделаем нормализация RMSnorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.norm(2, dim=-1, keepdim=True)
        rms = norm / (x.size(-1) ** 0.5)
        return self.scale * x / (rms + self.eps)

# Реализация RoPE 
def apply_rope(x):
    # x: (B, T, H, D)
    B, T, H, D = x.shape
    assert D % 2 == 0
    half = D // 2

    freqs = 1.0 / (10000 ** (torch.arange(0, half, device=x.device).float() / half))
    pos = torch.arange(T, device=x.device).float()
    angles = torch.einsum('i,j->ij', pos, freqs)
    sin, cos = angles.sin(), angles.cos()
    sin = sin[None, :, None, :]
    cos = cos[None, :, None, :]

    x1, x2 = x[..., :half], x[..., half:]
    x_rot = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return x_rot


class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size

    def forward(self, x):
        B, T, _ = x.shape
        H = 1
        D = self.head_size

        # produce q, k, v of shape (B, T, 1, D)
        q = self.query(x).view(B, T, H, D)
        k = self.key(x).view(B, T, H, D)
        v = self.value(x).view(B, T, H, D)

        q = apply_rope(q)
        k = apply_rope(k)

        att = torch.einsum('bthd,bshd->bhts', q, k) * (D ** -0.5)
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        out = torch.einsum('bhts,bshd->bthd', att, v).reshape(B, T, D)
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))
# Реализуем класс SwitchMoEFeedForward
class SwitchMoEFeedForward(nn.Module):
    def __init__(self, n_embd, n_experts=4):
        super().__init__()
        self.n_experts = n_experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(n_embd, 4 * n_embd),
                nn.ReLU(),
                nn.Linear(4 * n_embd, n_embd)
            ) for _ in range(n_experts)
        ])
        self.gate = nn.Linear(n_embd, n_experts)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)  # (B*T, C)
        gate_logits = self.gate(x_flat)  # (B*T, E)
        top1_idx = torch.argmax(gate_logits, dim=-1)  # (B*T,)
        one_hot = F.one_hot(top1_idx, num_classes=self.n_experts).float()  # (B*T, E)

        # Load balancing loss (Switch Transformer trick)
        probs = F.softmax(gate_logits, dim=-1)
        density = probs.mean(dim=0)  # (E,)
        density_proxy = one_hot.mean(dim=0)  # (E,)
        aux_loss = (density * density_proxy).sum() * self.n_experts

        # dispatch
        outputs = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            mask = (top1_idx == i)
            if mask.any():
                selected = x_flat[mask]
                expert_out = expert(selected)
                outputs[mask] = expert_out

        out = outputs.view(B, T, C)
        return self.dropout(out), aux_loss

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = SwitchMoEFeedForward(n_embd, n_experts=4)  # Moe
        self.ln1 = RMSNorm(n_embd)
        self.ln2 = RMSNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        ff_out, aux_loss = self.ffwd(self.ln2(x))  # получаем также aux_loss
        x = x + ff_out
        return x, aux_loss


class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx)
        aux_losses = []
        for block in self.blocks:
            x, aux_loss = block(x)
            aux_losses.append(aux_loss)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        total_aux_loss = torch.stack(aux_losses).mean()

        if targets is None:
            return logits, total_aux_loss
        else:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
            return logits, loss + 0.01 * total_aux_loss  # λ=0.01

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = CosineAnnealingLR(optimizer, T_max=max_iters)

best_val_loss = float('inf')
patience_counter = 0


##Добавим early stopping
for iter in range(max_iters):
    if iter % eval_interval == 0:
        print(f"[INFO] Starting evaluation at step {iter}")
        losses = estimate_loss()
        val_loss = losses['val']
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
            print("✅ New best model saved.")
        else:
            patience_counter += 1
            print(f"⚠️ No improvement. Patience left: {patience - patience_counter}")
            if patience_counter >= patience:
                print(f"⏹️ Early stopping at step {iter}")
                break

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

model.load_state_dict(torch.load('best_model.pt'))
model.eval()
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

14.242159 M parameters
[INFO] Starting evaluation at step 0


step 0: train loss 4.5553, val loss 4.4575
✅ New best model saved.
[INFO] Starting evaluation at step 100


step 100: train loss 2.6074, val loss 2.5530
✅ New best model saved.
[INFO] Starting evaluation at step 200


step 200: train loss 2.4425, val loss 2.3325
✅ New best model saved.
[INFO] Starting evaluation at step 300


step 300: train loss 2.3538, val loss 2.2717
✅ New best model saved.
[INFO] Starting evaluation at step 400


...

step 9400: train loss 1.5869, val loss 1.6671
⚠️ No improvement. Patience left: 3
[INFO] Starting evaluation at step 9500


step 9500: train loss 1.5816, val loss 1.6552
⚠️ No improvement. Patience left: 2
[INFO] Starting evaluation at step 9600


step 9600: train loss 1.5892, val loss 1.6682
⚠️ No improvement. Patience left: 1
[INFO] Starting evaluation at step 9700


C:\Users\User\AppData\Local\Temp\ipykernel_8728\2226253602.py:267: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pt'))


step 9700: train loss 1.5803, val loss 1.6799
⚠️ No improvement. Patience left: 0
⏹️ Early stopping at step 9700
	нимали, другой глаза и причессивный! Доллара на которой Еле. На страшной может, то - этот выйдем как белых, в славу приставьте! Крестьяне! Вот виграй - несль - ни вылялся. Первый глаз - мала! 9-й голод, бёла знакомый зря. Французский рыландой горящий петь - в семанских пот и смахнуть... 11. Пыть я, г. результат разразиденной поцелуешь спину. 21 Нельзя. Берет восставал: только без в ногу. А предлагал близкий человечеством. Должно быть из угольных джелезнах требуется в две такого, тоже наши 10 миллионов, шарем для первых, а не забудьте фатуризма лач лить-то руки, а и поле, музыкой барабан, дачу им. Стой в за коммуны твердо водки. Другой руки. Здесь Коммуны к Монфетя к сбатиле. У Пилфекторы, копил струнил. Но не были долгием капиталистам". и нешепту его пят меня улицу стрячный. Поэт больше Госадин гоудняет партиец стихов) так у нас молвится в зубережительной воды! Бандант же д